# Blending

Workbench for combining the individual experiment runs in `experiments/runs.csv` into
stronger predictions. Planned uses:

- **Ensembling** — simple/weighted averages of model probabilities (and rank averages).
- **Hill climbing** — greedily grow a weighted blend on the OOF predictions, adding the
  model that most improves the OOF metric at each step (Caruana-style ensemble selection).
- **Stacking** — train a meta-learner on the out-of-fold (OOF) prediction columns.

All of these need the per-run **out-of-fold predictions** (`oof_proba.npy`), which give a
leakage-free prediction for every training row under the shared
`StratifiedKFold(5, shuffle=True, random_state=42)` split.

This first cut just measures **how similar the runs are to each other**, pairwise, via the
Pearson and Spearman correlation of their OOF probabilities. Blending pays off most when the
members are individually strong *and* mutually **de-correlated**, so this is the natural
first diagnostic.

## What is Spearman's correlation coefficient?

**Pearson's** $r$ measures the strength of a *linear* relationship between two variables — it
works on the raw values and asks "do they move up and down together, proportionally?"

**Spearman's** $\rho$ (rho) is just **Pearson's $r$ computed on the ranks** of the values
instead of the values themselves. You replace each column by `1, 2, 3, …` according to sort
order, then correlate those ranks. As a result it measures whether the two variables move
together **monotonically** — when one goes up, does the other tend to go up — *regardless of
whether the relationship is a straight line*.

$\rho = 1$ means the two runs rank every customer in the **exact same order**; $\rho = -1$
means perfectly reversed; $0$ means no monotonic relationship.

Why it matters here:

- The competition metric (ROC-AUC) and rank-averaging both depend **only on the ordering** of
  the predicted probabilities, not their absolute calibration. Two models can disagree on the
  raw probabilities (low Pearson) yet rank customers near-identically (high Spearman) — and
  for an AUC blend, it is the **Spearman** agreement that tells you how much fresh signal a
  second model actually adds.
- Spearman is **invariant to any monotonic rescaling** (e.g. one model being systematically
  over-confident), so it is a cleaner diversity measure for ranking-based ensembles, while
  Pearson is the relevant one for plain probability averaging.

In [1]:
import os

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# The canonical training-set size. Runs whose OOF array is shorter were trained on a
# subsample (e.g. the TabPFN/TabICL 50k EDA passes) and their rows do NOT align with the
# full-set runs, so they cannot be correlated row-wise and are excluded below.
CANON_N = 594_194

runs = pd.read_csv("experiments/runs.csv").dropna(subset=["run_id"])
runs = runs[runs["status"] == "success"].reset_index(drop=True).sort_values(["oof_roc_auc"], ascending=False)

oof_cols, labels, kept = [], [], []
skipped = []
seen_tags = set()
for _, r in runs.iterrows():
    path = os.path.join(str(r["artifact_dir"]).replace("\\", "/"), "oof_proba.npy")
    proba = np.load(path)
    if proba.shape[0] != CANON_N:
        skipped.append((r["tag"], proba.shape[0]))
        continue
    # Disambiguate runs that share a tag (e.g. lr-pipeline on fe_v0 vs fe_v1).
    label = r["tag"] if r["tag"] not in seen_tags else f"{r['tag']}|{r['data_version']}"
    seen_tags.add(r["tag"])
    oof_cols.append(proba)
    labels.append(label)
    kept.append(r)

oof = pd.DataFrame(np.column_stack(oof_cols), columns=labels)
meta = pd.DataFrame(kept)[["tag", "model_class", "data_version", "oof_roc_auc", "run_id"]]
meta.insert(0, "label", labels)

print(f"OOF matrix: {oof.shape[0]:,} rows x {oof.shape[1]} runs")
if skipped:
    print("Excluded (subsample OOF, rows do not align):")
    for tag, n in skipped:
        print(f"  - {tag}: {n:,} rows")

OOF matrix: 594,194 rows x 37 runs
Excluded (subsample OOF, rows do not align):
  - tabpfn-eda: 50,000 rows
  - tabicl-eda: 50,000 rows


In [2]:
# Pairwise correlation matrices across the run OOF probabilities.
# Pearson: linear agreement on the raw probabilities (relevant to probability averaging).
# Spearman: rank agreement (relevant to AUC / rank-averaging diversity).
pearson = oof.corr(method="pearson")

rho, _ = spearmanr(oof.values)          # one rank-and-correlate pass over all columns
spearman = pd.DataFrame(rho, index=labels, columns=labels)

def show(corr, title):
    print(title)
    styled = (corr.style
              .background_gradient(cmap="RdYlGn_r", vmin=corr.values.min(), vmax=1.0)
              .format("{:.4f}"))
    return styled

show(spearman, "Spearman (rank) correlation of OOF probabilities")

Spearman (rank) correlation of OOF probabilities


,lgbm-catreg-fe-min3,xgb-coupled-fe-min3,ebm-optuna-fe_v4_native,xgb-coupled-fe-nocross5,lgbm-refined-best,xgb-optuna-acc,lgbm-refined-best-native,catboost-optuna-rocauc,catboost-gpu-fe-min3,xgb-optuna-rocauc,catboost-optuna-acc,lgbm-catreg-fe_v3-study,lgbm-optuna-rocauc,lgbm-optuna-acc,lgbm-catreg-fe-nocross5,catboost-gpu-fe-v3-full,xgb-refined-best-fe_v3,lgbm-refined-best-native-updated,ebm-optuna-fe_v0_native,lgbm-optuna-best,lgbm-refined-best-fe_v3-updated,lgbm-refined-best-fe_v3,lgbm-refined-fe_v3-study,lgbm-untuned,rf-optuna-fe_v4_native,tabpfn-subfold-bag3,tabicl-subfold-bag3,rf-optuna-fe_v0,realmlp-fe-min3,tabm-optuna-fe-min3,tabm-baseline,realmlp-baseline,et-optuna-fe_v4_native,et-optuna-fe_v0_native,lr-targetenc-crosses-native,lr-pipeline,lr-pipeline|fe_v1
lgbm-catreg-fe-min3,1.0000,0.9987,0.9946,0.9980,0.9970,0.9964,0.9970,0.9957,0.9963,0.9958,0.9958,0.9973,0.9964,0.9962,0.9981,0.9951,0.9975,0.9973,0.9943,0.9940,0.9955,0.9940,0.9952,0.9944,0.9877,0.9922,0.9912,0.9847,0.9888,0.9919,0.9914,0.9841,0.9842,0.9842,0.9786,0.9694,0.9694
xgb-coupled-fe-min3,0.9987,1.0000,0.9939,0.9988,0.9963,0.9972,0.9964,0.9958,0.9969,0.9966,0.9961,0.9972,0.9956,0.9956,0.9982,0.9955,0.9984,0.9971,0.9941,0.9931,0.9957,0.9943,0.9956,0.9950,0.9891,0.9932,0.9921,0.9860,0.9898,0.9934,0.9930,0.9851,0.9862,0.9861,0.9795,0.9701,0.9701
ebm-optuna-fe_v4_native,0.9946,0.9939,1.0000,0.9932,0.9941,0.9936,0.9934,0.9937,0.9929,0.9933,0.9936,0.9923,0.9931,0.9933,0.9936,0.9919,0.9926,0.9932,0.9951,0.9909,0.9916,0.9900,0.9912,0.9903,0.9821,0.9886,0.9872,0.9799,0.9862,0.9880,0.9877,0.9812,0.9793,0.9798,0.9754,0.9663,0.9662
xgb-coupled-fe-nocross5,0.9980,0.9988,0.9932,1.0000,0.9966,0.9960,0.9968,0.9960,0.9970,0.9954,0.9963,0.9971,0.9947,0.9947,0.9990,0.9964,0.9991,0.9975,0.9937,0.9931,0.9958,0.9942,0.9960,0.9952,0.9885,0.9939,0.9931,0.9852,0.9904,0.9934,0.9928,0.9857,0.9857,0.9857,0.9805,0.9712,0.9712
lgbm-refined-best,0.9970,0.9963,0.9941,0.9966,1.0000,0.9963,0.9982,0.9954,0.9954,0.9962,0.9955,0.9964,0.9968,0.9971,0.9971,0.9948,0.9961,0.9978,0.9952,0.9965,0.9948,0.9932,0.9940,0.9938,0.9859,0.9913,0.9902,0.9832,0.9879,0.9901,0.9901,0.9834,0.9824,0.9828,0.9777,0.9694,0.9694
xgb-optuna-acc,0.9964,0.9972,0.9936,0.9960,0.9963,1.0000,0.9953,0.9953,0.9952,0.9981,0.9954,0.9950,0.9964,0.9969,0.9954,0.9936,0.9957,0.9954,0.9944,0.9936,0.9935,0.9919,0.9930,0.9935,0.9860,0.9901,0.9889,0.9841,0.9872,0.9899,0.9904,0.9829,0.9833,0.9838,0.9771,0.9681,0.9681
lgbm-refined-best-native,0.9970,0.9964,0.9934,0.9968,0.9982,0.9953,1.0000,0.9945,0.9951,0.9953,0.9946,0.9968,0.9957,0.9962,0.9972,0.9945,0.9961,0.9983,0.9944,0.9955,0.9947,0.9935,0.9938,0.9933,0.9863,0.9910,0.9899,0.9827,0.9874,0.9899,0.9896,0.9827,0.9822,0.9824,0.9774,0.9692,0.9692
catboost-optuna-rocauc,0.9957,0.9958,0.9937,0.9960,0.9954,0.9953,0.9945,1.0000,0.9967,0.9948,0.9987,0.9939,0.9944,0.9946,0.9960,0.9956,0.9955,0.9953,0.9947,0.9915,0.9936,0.9913,0.9936,0.9944,0.9845,0.9922,0.9912,0.9823,0.9900,0.9912,0.9912,0.9852,0.9830,0.9838,0.9798,0.9707,0.9707
catboost-gpu-fe-min3,0.9963,0.9969,0.9929,0.9970,0.9954,0.9952,0.9951,0.9967,1.0000,0.9947,0.9971,0.9952,0.9941,0.9944,0.9968,0.9971,0.9966,0.9960,0.9937,0.9917,0.9944,0.9924,0.9944,0.9943,0.9864,0.9931,0.9921,0.9836,0.9901,0.9924,0.9920,0.9853,0.9843,0.9847,0.9802,0.9711,0.9711
xgb-optuna-rocauc,0.9958,0.9966,0.9933,0.9954,0.9962,0.9981,0.9953,0.9948,0.9947,1.0000,0.9949,0.9949,0.9963,0.9970,0.9951,0.9933,0.9951,0.9954,0.9943,0.9936,0.9932,0.9916,0.9926,0.9933,0.9860,0.9900,0.9888,0.9841,0.9869,0.9896,0.9900,0.9827,0.9830,0.9835,0.9770,0.9681,0.9681


In [3]:
show(pearson, "Pearson (linear) correlation of OOF probabilities")

Pearson (linear) correlation of OOF probabilities


,lgbm-catreg-fe-min3,xgb-coupled-fe-min3,ebm-optuna-fe_v4_native,xgb-coupled-fe-nocross5,lgbm-refined-best,xgb-optuna-acc,lgbm-refined-best-native,catboost-optuna-rocauc,catboost-gpu-fe-min3,xgb-optuna-rocauc,catboost-optuna-acc,lgbm-catreg-fe_v3-study,lgbm-optuna-rocauc,lgbm-optuna-acc,lgbm-catreg-fe-nocross5,catboost-gpu-fe-v3-full,xgb-refined-best-fe_v3,lgbm-refined-best-native-updated,ebm-optuna-fe_v0_native,lgbm-optuna-best,lgbm-refined-best-fe_v3-updated,lgbm-refined-best-fe_v3,lgbm-refined-fe_v3-study,lgbm-untuned,rf-optuna-fe_v4_native,tabpfn-subfold-bag3,tabicl-subfold-bag3,rf-optuna-fe_v0,realmlp-fe-min3,tabm-optuna-fe-min3,tabm-baseline,realmlp-baseline,et-optuna-fe_v4_native,et-optuna-fe_v0_native,lr-targetenc-crosses-native,lr-pipeline,lr-pipeline|fe_v1
lgbm-catreg-fe-min3,1.0000,0.9985,0.9955,0.9979,0.9979,0.9968,0.9982,0.9965,0.9969,0.9966,0.9966,0.9976,0.9977,0.9977,0.9984,0.9964,0.9971,0.9980,0.9957,0.9963,0.9963,0.9960,0.9961,0.9941,0.9919,0.9901,0.9900,0.9898,0.9902,0.9889,0.9884,0.9883,0.9880,0.9857,0.9758,0.9678,0.9678
xgb-coupled-fe-min3,0.9985,1.0000,0.9950,0.9991,0.9972,0.9978,0.9976,0.9968,0.9976,0.9976,0.9971,0.9975,0.9969,0.9970,0.9984,0.9970,0.9984,0.9979,0.9958,0.9953,0.9965,0.9962,0.9967,0.9950,0.9933,0.9915,0.9914,0.9911,0.9916,0.9906,0.9902,0.9898,0.9898,0.9874,0.9771,0.9691,0.9691
ebm-optuna-fe_v4_native,0.9955,0.9950,1.0000,0.9944,0.9948,0.9945,0.9946,0.9948,0.9938,0.9942,0.9947,0.9933,0.9943,0.9945,0.9946,0.9936,0.9934,0.9943,0.9965,0.9930,0.9929,0.9923,0.9926,0.9910,0.9878,0.9881,0.9878,0.9861,0.9880,0.9859,0.9855,0.9861,0.9846,0.9827,0.9753,0.9675,0.9675
xgb-coupled-fe-nocross5,0.9979,0.9991,0.9944,1.0000,0.9975,0.9970,0.9980,0.9963,0.9973,0.9968,0.9967,0.9977,0.9962,0.9963,0.9988,0.9973,0.9988,0.9981,0.9954,0.9957,0.9966,0.9963,0.9968,0.9946,0.9933,0.9916,0.9914,0.9908,0.9916,0.9905,0.9900,0.9898,0.9897,0.9872,0.9771,0.9692,0.9692
lgbm-refined-best,0.9979,0.9972,0.9948,0.9975,1.0000,0.9966,0.9986,0.9964,0.9961,0.9966,0.9966,0.9971,0.9976,0.9978,0.9982,0.9963,0.9967,0.9981,0.9960,0.9978,0.9963,0.9959,0.9960,0.9942,0.9909,0.9904,0.9902,0.9894,0.9904,0.9888,0.9886,0.9888,0.9879,0.9858,0.9767,0.9689,0.9689
xgb-optuna-acc,0.9968,0.9978,0.9945,0.9970,0.9966,1.0000,0.9960,0.9970,0.9959,0.9988,0.9972,0.9956,0.9975,0.9977,0.9962,0.9953,0.9963,0.9959,0.9957,0.9950,0.9945,0.9941,0.9944,0.9944,0.9904,0.9887,0.9885,0.9902,0.9889,0.9876,0.9881,0.9883,0.9880,0.9864,0.9759,0.9679,0.9679
lgbm-refined-best-native,0.9982,0.9976,0.9946,0.9980,0.9986,0.9960,1.0000,0.9959,0.9966,0.9960,0.9962,0.9975,0.9967,0.9971,0.9987,0.9967,0.9971,0.9987,0.9954,0.9970,0.9966,0.9964,0.9963,0.9938,0.9914,0.9905,0.9903,0.9890,0.9905,0.9890,0.9884,0.9885,0.9877,0.9854,0.9763,0.9684,0.9684
catboost-optuna-rocauc,0.9965,0.9968,0.9948,0.9963,0.9964,0.9970,0.9959,1.0000,0.9962,0.9970,0.9988,0.9951,0.9965,0.9968,0.9962,0.9957,0.9955,0.9959,0.9962,0.9947,0.9945,0.9940,0.9944,0.9936,0.9901,0.9894,0.9893,0.9891,0.9895,0.9879,0.9880,0.9883,0.9875,0.9856,0.9763,0.9685,0.9685
catboost-gpu-fe-min3,0.9969,0.9976,0.9938,0.9973,0.9961,0.9959,0.9966,0.9962,1.0000,0.9958,0.9966,0.9963,0.9954,0.9958,0.9970,0.9974,0.9967,0.9966,0.9947,0.9945,0.9953,0.9951,0.9954,0.9932,0.9918,0.9901,0.9900,0.9895,0.9901,0.9891,0.9885,0.9882,0.9880,0.9856,0.9755,0.9677,0.9677
xgb-optuna-rocauc,0.9966,0.9976,0.9942,0.9968,0.9966,0.9988,0.9960,0.9970,0.9958,1.0000,0.9972,0.9954,0.9972,0.9976,0.9961,0.9952,0.9960,0.9959,0.9957,0.9949,0.9945,0.9941,0.9943,0.9944,0.9903,0.9889,0.9887,0.9900,0.9890,0.9877,0.9882,0.9883,0.9879,0.9862,0.9760,0.9681,0.9681


### Most de-correlated pairs

The pairs with the **lowest** Spearman correlation are the most promising blend partners —
they rank customers most differently, so each carries signal the other misses. (Pair them
with the leaderboard OOF ROC-AUC to balance diversity against individual strength.)

In [4]:
def ranked_pairs(corr, ascending=True, n=15):
    m = corr.values.copy()
    iu = np.triu_indices_from(m, k=1)
    pairs = pd.DataFrame({
        "run_a": [corr.index[i] for i in iu[0]],
        "run_b": [corr.columns[j] for j in iu[1]],
        "spearman": m[iu],
    })
    pairs = pairs.sort_values("spearman", ascending=ascending).reset_index(drop=True)
    return pairs.head(n)

least = ranked_pairs(spearman, ascending=True, n=15)
least["spearman"] = least["spearman"].round(4)
least.style.hide(axis="index")

run_a,run_b,spearman
rf-optuna-fe_v0,lr-pipeline|fe_v1,0.961000
rf-optuna-fe_v0,lr-pipeline,0.961000
rf-optuna-fe_v4_native,lr-pipeline|fe_v1,0.963400
rf-optuna-fe_v4_native,lr-pipeline,0.963400
lgbm-optuna-best,lr-pipeline|fe_v1,0.964100
lgbm-optuna-best,lr-pipeline,0.964100
lgbm-refined-best-fe_v3,lr-pipeline|fe_v1,0.965700
lgbm-refined-best-fe_v3,lr-pipeline,0.965700
ebm-optuna-fe_v4_native,lr-pipeline|fe_v1,0.966200
ebm-optuna-fe_v4_native,lr-pipeline,0.966300


## Selecting which runs to blend — and which to leave out

Three questions decide the candidate set: do we blend **all** the runs; if not, **how**
do we filter (beyond "pick de-correlated models"); and **how many of each kind** of model
do we keep?

**Short answer: no — we do not blend all 37 aligned runs.** A blend pays off when its members
are individually strong *and* mutually **de-correlated**. The library badly violates the second
condition: 24 of the 37 runs are gradient-boosted trees (13 LightGBM, 5 XGBoost, 4 CatBoost,
2 EBM) — a few *engines re-tuned across feature-engineering versions*. Their pairwise Spearman
sits at **0.99+**: they are not 24 independent opinions, they are a handful of opinions measured
many times over. Throwing all of them in causes three distinct failures:

1. **Majority-by-count bias** — an equal-weight average collapses to "whatever the LightGBMs
   agree on," because 13 of 37 votes are near-identical.
2. **Multicollinearity** — a linear stacker handed 24 columns at ρ > 0.99 returns wild,
   high-variance coefficients.
3. **Overfitting the combiner** — every extra near-duplicate column is one more knob a
   greedy or learned blender can use to fit *OOF noise* instead of signal.

So we filter, in three passes:

**Filter 1 — Eligibility.** Only the full-set OOF runs (594,194 rows) align row-wise. The two
50k TabPFN/TabICL EDA passes were already dropped in the load cell; their full-set bagged
descendants stay.

**Filter 2 — De-duplicate *within an engine* (this is the filter beyond raw diversity).**
Diversity is a *pairwise* property; redundancy is what this pass removes. Within each engine we
keep **only the single best-OOF-AUC tuning** and discard the rest. It is a
strength-conditioned-on-redundancy rule: among runs that rank customers almost identically, keep
the strongest — it dominates the others on the only axis where they differ (a sliver of AUC) at
**zero diversity cost**. This collapses the 24 GBDT runs to 4, and the duplicate FE/seed
variants of every other learner to one each. (Note this dedups *re-tunings of the same engine*;
two **different** engines are always kept, even at high ρ — see the GBDT note below.)

**Filter 3 — Let the combiner judge the weak-but-diverse members.** We do **not** pre-drop a
model just for being weak. Logistic regression is the weakest single learner (0.908) yet the
**most de-correlated run in the whole library** (mean ρ ≈ 0.97) — precisely the orthogonal signal
a blend wants. Rather than guess, keep it and let the weighting/stacking step decide: hill
climbing and a regularized stacker shrink to ~0 anything that does not earn its place. How much
curation you *need* actually depends on the combiner — an equal-weight mean is fragile and
demands aggressive pruning, whereas strength-weighting, hill climbing and a regularized stack are
robust to leftover redundancy.

### How many of each, and which?

One representative **per engine**, chosen by best OOF ROC-AUC — *except* the linear slot, picked
by **diversity** (the most de-correlated logistic run), because a linear model will never win on
standalone AUC here and its entire value is orthogonality. That yields **11 candidates across 5
broad families**:

| family | kept | why |
|---|---|---|
| **Boosted trees** | **4** — LightGBM, XGBoost, CatBoost, EBM | distinct inductive bias per engine (leaf-wise histogram vs depth/loss-guided vs ordered boosting vs additive shape functions), so cross-engine ρ ≈ 0.994–0.996 sits below intra-engine ρ ≈ 0.999; averaging *different libraries* is a reliable variance-reducer. We keep one best **tuning** of each and drop the other 20. LightGBM↔XGBoost are themselves ρ ≈ 0.999 near-twins — we leave both in and let the combiner split their weight rather than hand-drop a top-2 model. |
| **Bagged trees** | **2** — RandomForest, ExtraTrees | averaging-based trees, ρ ≈ 0.98 to the boosters — a genuinely different error structure. |
| **Deep nets** | **2** — RealMLP, TabM | distinct tabular-MLP architectures, ρ ≈ 0.99. |
| **In-context transformers** | **2** — TabPFN, TabICL | ρ ≈ 0.999 twins; like LGBM/XGB we keep both and let the combiner handle the redundancy. |
| **Linear** | **1** — logistic regression | the single most de-correlated run (mean ρ ≈ 0.97); selected on diversity, not AUC. |

The cell below builds this pool programmatically straight from `meta`, so the rule is auditable
rather than a hand-typed list.

In [5]:
from sklearn.metrics import roc_auc_score

# --- targets + the canonical fold map that produced every base run's OOF ---
y = pd.read_parquet("data/processed/train_df_fe_v0.parquet")["Churn"].to_numpy()
folds = (pd.read_csv("experiments/cv_folds_seed42.csv.gz")
           .sort_values("id")["fold"].to_numpy())
assert len(y) == CANON_N and len(folds) == CANON_N

# Re-assert every kept run is row-aligned: its OOF must reproduce the logged ROC-AUC
# (same guard as scripts/check_oof_alignment.py). A misaligned column would silently
# poison every blend below.
for lab in labels:
    rid = meta.loc[meta["label"] == lab, "run_id"].iloc[0]
    logged = float(meta.loc[meta["label"] == lab, "oof_roc_auc"].iloc[0])
    assert abs(roc_auc_score(y, oof[lab].to_numpy()) - logged) < 1e-9, f"{lab} not aligned"
print(f"All {oof.shape[1]} runs row-aligned to the canonical OOF order "
      f"(prevalence = {y.mean():.4f}).")

# --- map model_class -> (broad family, engine) and attach a diversity score ---
TAXONOMY = {
    "LGBMClassifier":                ("Boosted trees", "LightGBM"),
    "XGBClassifier":                 ("Boosted trees", "XGBoost"),
    "CatBoostClassifier":            ("Boosted trees", "CatBoost"),
    "ExplainableBoostingClassifier": ("Boosted trees", "EBM"),
    "RandomForestClassifier":        ("Bagged trees",  "RandomForest"),
    "ExtraTreesClassifier":          ("Bagged trees",  "ExtraTrees"),
    "RealMLP_TD_Classifier":         ("Deep nets",      "RealMLP"),
    "SeedEnsembleRealMLP":           ("Deep nets",      "RealMLP"),
    "TabM_D_Classifier":             ("Deep nets",      "TabM"),
    "SeedEnsembleTabM":              ("Deep nets",      "TabM"),
    "BaggedSubsampleTabPFN":         ("In-context",     "TabPFN"),
    "BaggedSubsampleTabICL":         ("In-context",     "TabICL"),
    "Pipeline":                      ("Linear",         "Logistic"),
}
tax = meta["model_class"].map(TAXONOMY)
meta["family"] = tax.str[0]
meta["engine"] = tax.str[1]
# mean Spearman to all *other* runs (low = carries the most fresh ranking signal)
_off = spearman.where(~np.eye(len(spearman), dtype=bool))
meta["mean_rho"] = meta["label"].map(_off.mean())

# --- Filter 2: best-AUC run per engine; Filter 3 exception: linear slot on diversity ---
best_per_engine = meta.loc[meta.groupby("engine")["oof_roc_auc"].idxmax()]
linear_keep = meta.loc[meta["engine"] == "Logistic", "mean_rho"].idxmin()
keep_idx = [i for i in best_per_engine.index if meta.loc[i, "engine"] != "Logistic"]
keep_idx.append(linear_keep)
curated_meta = meta.loc[keep_idx].sort_values("oof_roc_auc", ascending=False).reset_index(drop=True)
CURATED = curated_meta["label"].tolist()
FULL = list(oof.columns)

print(f"\nCurated pool: {len(CURATED)} runs (from {len(FULL)} aligned) — "
      f"{curated_meta['family'].nunique()} families, {curated_meta['engine'].nunique()} engines\n")
print(curated_meta[["label", "family", "engine", "oof_roc_auc", "mean_rho"]]
      .to_string(index=False, formatters={"oof_roc_auc": "{:.6f}".format,
                                          "mean_rho": "{:.3f}".format}))
print("\nKept per family:")
print(curated_meta["family"].value_counts().to_string())

All 37 runs row-aligned to the canonical OOF order (prevalence = 0.2252).

Curated pool: 11 runs (from 37 aligned) — 5 families, 11 engines

                  label        family       engine oof_roc_auc mean_rho
    lgbm-catreg-fe-min3 Boosted trees     LightGBM    0.916685    0.992
    xgb-coupled-fe-min3 Boosted trees      XGBoost    0.916650    0.992
ebm-optuna-fe_v4_native Boosted trees          EBM    0.916644    0.988
 catboost-optuna-rocauc Boosted trees     CatBoost    0.916498    0.991
 rf-optuna-fe_v4_native  Bagged trees RandomForest    0.914659    0.984
    tabpfn-subfold-bag3    In-context       TabPFN    0.914245    0.990
    tabicl-subfold-bag3    In-context       TabICL    0.914207    0.990
        realmlp-fe-min3     Deep nets      RealMLP    0.913991    0.987
    tabm-optuna-fe-min3     Deep nets         TabM    0.913728    0.990
 et-optuna-fe_v4_native  Bagged trees   ExtraTrees    0.913339    0.983
      lr-pipeline|fe_v1        Linear     Logistic    0.907934    0

## 1. Weighted ensembling

The simplest combiners: fixed, **rule-based** weights over the member probabilities — no fitting
on the labels beyond reading each model's strength. We try four schemes and score every blend by
the same leakage-free OOF ROC-AUC the base models were judged on.

- **Equal-weight mean (all 37)** — the naive "average everything" baseline.
- **Equal-weight mean (curated 11)** — same scheme on the de-duplicated pool.
- **Rank mean (curated)** — average the *ranks* instead of the probabilities. Because ROC-AUC
  depends only on ordering, rank-averaging is the calibration-free way to average for AUC and is
  immune to one member being systematically over-confident.
- **Strength-weighted mean** — weight each member by `softmax(AUC / T)`. The temperature `T`
  interpolates between equal weights (large `T`) and "all weight on the single best model"
  (`T → 0`); we sweep a few values.

The instructive result is that **equal-weighting is a trap here**: averaging 37 highly-correlated
members (most of them GBDT clones) lands *below the single best model*, because the weak/duplicate
members get an equal vote. Concentrating weight on the strong models — while keeping a sliver of
diversity — is what actually clears the best single run.

In [6]:
from scipy.stats import rankdata

RESULTS = {}                       # strategy name -> OOF ROC-AUC, filled across the segments
def blend_auc(p): return roc_auc_score(y, np.asarray(p))
def P(cols):      return oof[cols].to_numpy()

# equal-weight means
RESULTS["mean · all 37"]      = blend_auc(P(FULL).mean(axis=1))
RESULTS["mean · curated 11"]  = blend_auc(P(CURATED).mean(axis=1))
# rank mean on the curated pool
ranks = np.column_stack([rankdata(oof[c].to_numpy()) for c in CURATED])
RESULTS["rank-mean · curated"] = blend_auc(ranks.mean(axis=1))
# strength (softmax-AUC) weighting on the curated pool
aucs = np.array([blend_auc(oof[c].to_numpy()) for c in CURATED])
for T in (0.01, 0.005, 0.002, 0.001):
    w = np.exp((aucs - aucs.max()) / T); w /= w.sum()
    RESULTS[f"softmax(AUC, T={T})"] = blend_auc((P(CURATED) * w).sum(axis=1))

best_single = curated_meta["oof_roc_auc"].max()
tbl = (pd.Series(RESULTS, name="oof_roc_auc").to_frame()
         .assign(**{"vs best single": lambda d: d["oof_roc_auc"] - best_single}))
print(f"best single model (curated): {best_single:.6f}\n")
print(tbl.to_string(formatters={"oof_roc_auc": "{:.6f}".format,
                                "vs best single": "{:+.6f}".format}))

best single model (curated): 0.916685

                      oof_roc_auc vs best single
mean · all 37            0.916451      -0.000234
mean · curated 11        0.915947      -0.000738
rank-mean · curated      0.915899      -0.000786
softmax(AUC, T=0.01)     0.916223      -0.000462
softmax(AUC, T=0.005)    0.916422      -0.000263
softmax(AUC, T=0.002)    0.916783      +0.000098
softmax(AUC, T=0.001)    0.916990      +0.000305


## 2. Hill climbing (Caruana ensemble selection)

Rather than impose a weighting rule, **learn** a sparse, non-negative weighting greedily on the
OOF predictions — Caruana et al.'s *ensemble selection* (2004):

1. Start the ensemble from the single best member.
2. At each step, try adding (**with replacement**) each candidate and keep the one whose
   inclusion most improves the OOF ROC-AUC of the running *average*.
3. Stop when no addition helps. A model's weight is simply **how many times it was picked**.

Selection-with-replacement is what turns counts into real-valued weights, and optimising the
metric directly means hill climbing can exploit complementary errors that a strength-only
weighting cannot. Its risk is **overfitting the OOF set** — with many interchangeable correlated
columns it can chase noise. The standard guard (used here) is **bagging the ensemble selection**:
repeat the greedy build on random subsets of the candidate pool and average the selection
frequencies, which stabilises the weights.

We run it on the **curated** pool (the honest estimate) and, for contrast, on the **full 37**.
The full-pool blend usually shows a *higher* OOF AUC — but it spreads weight across many
near-duplicate GBDT re-tunings, which is exactly the OOF-overfitting signature: extra apparent
gain bought with redundant columns that will not generalise. The curated number is the one to
trust.

In [7]:
def hill_climb(cols, n_iter=50, n_bags=25, frac=0.5, seed=42):
    "Bagged Caruana ensemble selection on OOF probabilities. Returns (weights, oof_auc)."
    M = oof[cols].to_numpy()
    m = len(cols)
    rng = np.random.RandomState(seed)
    freq = np.zeros(m)
    for _ in range(n_bags):
        sub = rng.choice(m, size=max(2, int(frac * m)), replace=False)
        start = max(sub, key=lambda j: blend_auc(M[:, j]))   # best member in this bag
        ens = M[:, start].copy(); counts = np.zeros(m); counts[start] = 1
        n = 1; cur = blend_auc(ens)
        for _ in range(n_iter):
            best_j, best_s = None, cur
            for j in sub:
                s = blend_auc((ens + M[:, j]) / (n + 1))
                if s > best_s:
                    best_s, best_j = s, j
            if best_j is None:
                break
            ens += M[:, best_j]; counts[best_j] += 1; n += 1; cur = best_s
        freq += counts / counts.sum()
    w = freq / freq.sum()
    return w, blend_auc(M @ w)

w_cur, auc_cur = hill_climb(CURATED)
w_full, auc_full = hill_climb(FULL)
RESULTS["hill-climb · curated"] = auc_cur
RESULTS["hill-climb · full 37"] = auc_full

print(f"hill-climb (curated 11): OOF ROC-AUC = {auc_cur:.6f}")
print(f"hill-climb (full 37):    OOF ROC-AUC = {auc_full:.6f}   "
      f"(higher, but spread over redundant GBDT clones → OOF-overfit)\n")

wc = (pd.Series(w_cur, index=CURATED).sort_values(ascending=False))
wc = wc[wc > 1e-4]
print("Curated blend weights (selection frequency):")
print(wc.to_frame("weight").join(curated_meta.set_index("label")["family"])
        .to_string(formatters={"weight": "{:.3f}".format}))

hill-climb (curated 11): OOF ROC-AUC = 0.917062
hill-climb (full 37):    OOF ROC-AUC = 0.917093   (higher, but spread over redundant GBDT clones → OOF-overfit)

Curated blend weights (selection frequency):
                        weight         family
ebm-optuna-fe_v4_native  0.353  Boosted trees
xgb-coupled-fe-min3      0.230  Boosted trees
lgbm-catreg-fe-min3      0.223  Boosted trees
catboost-optuna-rocauc   0.153  Boosted trees
rf-optuna-fe_v4_native   0.027   Bagged trees
tabicl-subfold-bag3      0.013     In-context


## 3. Stacking / "tracking" (a meta-learner on the OOF columns)

Stacking trains a **meta-learner** whose features are the base models' predictions and whose
target is the true label. The trap is leakage: if the meta-learner is fit and scored on the same
rows, it sees each base model's prediction for a row that base model may effectively have
memorised. We avoid it the same way the base OOF was built — **cross-validate the meta-learner
over the identical `seed-42` fold map** (`experiments/cv_folds_seed42.csv.gz`). For each fold the
meta-model trains on the other four folds' OOF rows and predicts the held-out fold, so every
stacked prediction is genuinely out-of-fold. This matches `scripts/_blend_preview.py`.

We try three meta-learners on the curated pool, as requested:

1. **Logistic regression** on the **logit** of each probability (the natural scale for combining
   probabilistic models — raw probabilities double-squash through the sigmoid), features
   standardised, L2-regularised. A linear stacker can assign *unconstrained* (incl. negative)
   coefficients, so it can correct systematic biases that a non-negative weighting cannot.
2. **Untuned LightGBM** — library defaults — to see whether non-linear *interactions* between
   base models help (e.g. "trust the net only when the boosters disagree").
3. **A 50-trial LightGBM Optuna study** — the same meta-LGBM with its hyper-parameters tuned.

A useful prior: on a meta-matrix this **narrow and collinear** (11 columns, all ρ > 0.96) a tree
stacker has little room to find genuine interactions and tends to overfit OOF noise, so the
**untuned** LGBM often lands *below the best single model*; tuning toward shallow, heavily
regularised trees claws most of that back, while the linear stack is typically the strongest.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def stacked_oof(cols, make_model, use_logit=False, standardize=False):
    "OOF ROC-AUC of a meta-learner, CV'd over the canonical seed-42 folds."
    X = logit(oof[cols].to_numpy()) if use_logit else oof[cols].to_numpy()
    meta_pred = np.zeros(len(y))
    for f in range(5):
        tr, va = folds != f, folds == f
        Xtr, Xva = X[tr], X[va]
        if standardize:
            sc = StandardScaler().fit(Xtr)
            Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
        model = make_model()
        model.fit(Xtr, y[tr])
        meta_pred[va] = model.predict_proba(Xva)[:, 1]
    return blend_auc(meta_pred)

RESULTS["stack · logistic (logit)"] = stacked_oof(
    CURATED, lambda: LogisticRegression(C=1.0, max_iter=2000),
    use_logit=True, standardize=True)
RESULTS["stack · untuned LGBM"] = stacked_oof(
    CURATED, lambda: lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))

print(f"stack · logistic (logit feats): {RESULTS['stack · logistic (logit)']:.6f}")
print(f"stack · untuned LGBM:           {RESULTS['stack · untuned LGBM']:.6f}")

d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


stack · logistic (logit feats): 0.917098
stack · untuned LGBM:           0.916576


d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 50-trial study. Objective = the same leakage-free OOF stack AUC over the seed-42 folds,
# so the reported best value is directly comparable to the two meta-learners above. The space
# is deliberately shallow + strongly regularised: the meta-problem is tiny (11 collinear
# features) and the real danger is overfitting the meta-layer, not underfitting it.
Xc = oof[CURATED].to_numpy()

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 100, 600),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 7, 63),
        max_depth        = trial.suggest_int("max_depth", 2, 6),
        min_child_samples= trial.suggest_int("min_child_samples", 20, 500),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq   = 1,
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        random_state=42, n_jobs=-1, verbose=-1,
    )
    pred = np.zeros(len(y))
    for f in range(5):
        tr, va = folds != f, folds == f
        mdl = lgb.LGBMClassifier(**params).fit(Xc[tr], y[tr])
        pred[va] = mdl.predict_proba(Xc[va])[:, 1]
    return blend_auc(pred)

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=False)
RESULTS["stack · LGBM (Optuna 50)"] = study.best_value

print(f"stack · LGBM (Optuna, 50 trials): {study.best_value:.6f}")
print(f"  improvement over untuned LGBM:  {study.best_value - RESULTS['stack · untuned LGBM']:+.6f}")
print("  best params:")
for k, v in study.best_params.items():
    print(f"    {k:18s} {v}")

d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\My Code\Kaggle projects\Predict Customer Churn\.venv\Lib\site-packages\sklearn\utils\validation.py:28

stack · LGBM (Optuna, 50 trials): 0.916997
  improvement over untuned LGBM:  +0.000421
  best params:
    n_estimators       297
    learning_rate      0.011986174662491458
    num_leaves         49
    max_depth          5
    min_child_samples  171
    subsample          0.6101163013268003
    colsample_bytree   0.5262293898988325
    reg_lambda         0.3478073156596913


## Summary — and producing the blended test predictions

Collecting every strategy on the one leakage-free yardstick (OOF ROC-AUC over the seed-42 folds).
The pattern is consistent with the diagnostics above: **equal-weighting and the untuned tree
stacker fall below the best single model**, while strength-weighting, hill climbing and the
**logistic stack** clear it — the linear meta-learner typically winning by a hair because it can
use unconstrained coefficients on the logit-scale features.

The final cell refits the **winning** combiner on *all* OOF rows and applies it to the matching
`test_proba_mean.npy` columns to produce the blended test-set probabilities, alongside the
robust hill-climb blend for comparison. (We deliberately do not overwrite any submission file —
the save line is left commented.)

In [10]:
summary = (pd.Series(RESULTS, name="oof_roc_auc").sort_values(ascending=False).to_frame()
             .assign(**{"vs best single": lambda d: d["oof_roc_auc"] - best_single}))
print(f"best single model (curated): {best_single:.6f}\n")
print(summary.to_string(formatters={"oof_roc_auc": "{:.6f}".format,
                                     "vs best single": "{:+.6f}".format}))
winner = summary.index[0]
print(f"\nWinner: {winner}  ({summary.iloc[0, 0]:.6f})")

best single model (curated): 0.916685

                         oof_roc_auc vs best single
stack · logistic (logit)    0.917098      +0.000413
hill-climb · full 37        0.917093      +0.000408
hill-climb · curated        0.917062      +0.000377
stack · LGBM (Optuna 50)    0.916997      +0.000312
softmax(AUC, T=0.001)       0.916990      +0.000305
softmax(AUC, T=0.002)       0.916783      +0.000098
stack · untuned LGBM        0.916576      -0.000109
mean · all 37               0.916451      -0.000234
softmax(AUC, T=0.005)       0.916422      -0.000263
softmax(AUC, T=0.01)        0.916223      -0.000462
mean · curated 11           0.915947      -0.000738
rank-mean · curated         0.915899      -0.000786

Winner: stack · logistic (logit)  (0.917098)


In [11]:
# Assemble the aligned test meta-matrix (same column order as CURATED).
# Key on run_id (unique) — the lr-pipeline *tag* is shared by two runs.
runs_by_id = pd.read_csv("experiments/runs.csv").dropna(subset=["run_id"]).set_index("run_id")
def test_col(label):
    rid = meta.loc[meta["label"] == label, "run_id"].iloc[0]
    d = str(runs_by_id.loc[rid, "artifact_dir"]).replace("\\", "/")
    return np.load(f"{d}/test_proba_mean.npy")

test_X = np.column_stack([test_col(c) for c in CURATED])
assert test_X.shape[0] == 254_655 and test_X.shape[1] == len(CURATED)

# (a) Logistic stack refit on ALL oof rows -> applied to test (the OOF winner).
sc = StandardScaler().fit(logit(oof[CURATED].to_numpy()))
lr = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(logit(oof[CURATED].to_numpy())), y)
test_stack = lr.predict_proba(sc.transform(logit(test_X)))[:, 1]

# (b) Hill-climb convex blend (robust / simple deployment choice).
test_hill = test_X @ w_cur

test_ids = pd.read_parquet("data/processed/test_df_fe_v0.parquet")["id"].to_numpy()
submission = pd.DataFrame({"id": test_ids, "Churn": test_stack})
print("Blended TEST predictions (logistic stack):")
print(submission.head().to_string(index=False))
print(f"\nstack vs hill-climb agreement on test: "
      f"Spearman {spearmanr(test_stack, test_hill).statistic:.5f}, "
      f"Pearson {np.corrcoef(test_stack, test_hill)[0,1]:.5f}")
# submission.to_csv("submission_blend.csv", ibndex=False)  # uncomment to write

Blended TEST predictions (logistic stack):
    id    Churn
594194 0.064691
594195 0.000667
594196 0.099485
594197 0.003140
594198 0.526337

stack vs hill-climb agreement on test: Spearman 0.99990, Pearson 0.99986
